# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`
This notebook provides a guided template for loading and exploring the FAIR² dataset using the `mlcroissant` library. We'll walk through data loading, schema inspection, data extraction, analysis, and visualization, referencing all field, record set, and column entities by their `@id`s for traceability and reproducibility.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -U mlcroissant

## 1. Data Loading
Load metadata and records from the FAIR² dataset using `mlcroissant`. We'll use the Croissant schema URL, and examine dataset metadata including its name and description.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset Name: {metadata.name}")
print(f"Description: {metadata.description}\n")

## 2. Data Overview
Let's inspect the data structure, including available record sets, fields, and columns, referencing them by their `@id`s.

In [ ]:
# List available record sets and some details
record_sets = list(dataset.record_sets)
print("Available Record Sets (by @id):")
for rs in record_sets:
    print(f"  - {rs['@id']}: {rs.get('name', '')}")

# For each record set, show the available fields (@id) and columns (@id where possible)
for rs in record_sets:
    print(f"\nRecord Set: {rs['@id']}")
    fields = rs.get('field', [])
    if isinstance(fields, dict):
        fields = [fields]
    print("  Fields:")
    for f in fields:
        if isinstance(f, dict):
            print(f"    - {f.get('@id')}, type: {f.get('dataType', '')}, description: {f.get('description', '')}")
        elif isinstance(f, str):
            print(f"    - {f}")
    if 'column' in rs:
        cols = rs['column']
        if isinstance(cols, list):
            print("  Columns:")
            for col in cols:
                if isinstance(col, dict):
                    print(f"    - {col.get('@id')}")
                else:
                    print(f"    - {col}")

## 3. Data Extraction
We now load data from the record set(s) into DataFrames. 
All record sets, fields, and columns are referenced by their `@id`. If your dataset has more than one record set, you can extract them all by iterating over the discovered `@id`s.

In [ ]:
# Gather all record set @id's for extraction
record_set_ids = [rs['@id'] for rs in dataset.record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    # Load all records for this record set, referencing it by @id
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded {len(df)} records from record set {record_set_id}")

# Display columns of the first record set as an example
if record_set_ids:
    main_rs = record_set_ids[0]
    print(f"\nFields (columns) in record set '{main_rs}':")
    print(dataframes[main_rs].columns.tolist())
    display(dataframes[main_rs].head())

## 4. Exploratory Data Analysis (EDA)
Let's perform basic EDA: filter records, normalize fields, and group by key attributes, referencing the relevant field or column by its `@id`.

> **Note:** Please adjust `numeric_field_id` and `group_field_id` below to match an actual field or column `@id` from your record set. Consult above printouts if necessary.

In [ ]:
# Example: Adjust these variable to actual field @id from the dataset (see section 2 output)
record_set_id = main_rs  # Use the main record set
df = dataframes[record_set_id]

# As an example, let's try with a common numerical column name (replace if needed)
numeric_field_id = None
for col in df.columns:
    # Pick a likely numeric column
    if ('coef' in col.lower() or 'log_likelihood' in col.lower() or 'std' in col.lower() or 'age' in col.lower()):
        numeric_field_id = col
        break
if numeric_field_id is None:
    print('Could not guess a numeric field, please specify manually.')
else:
    print(f"Using numeric field: {numeric_field_id}")
    # Ensure the column is numeric
    df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')

    # Set threshold for filtering
    threshold = df[numeric_field_id].quantile(0.75)  # 75th percentile as an example
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold:.2f} (top 25%):")
    print(filtered_df.head())

    # Normalize the field
    norm_col = f"{numeric_field_id}_normalized"
    filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"\nNormalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, norm_col]].head())

    # Try grouping by a likely categorical attribute
    group_field_id = None
    for col in df.columns:
        if col != numeric_field_id and (('gender' in col.lower()) or ('ward' in col.lower()) or ('county' in col.lower()) or ('group' in col.lower()) ):
            group_field_id = col
            break
    if group_field_id:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
        print(f"\nMean {numeric_field_id} grouped by {group_field_id}:")
        print(grouped_df.head())
    else:
        print('Could not find a suitable group field.')

## 5. Visualization
Let's visualize the distribution of the chosen numeric field, and (if a group field is present) compare means across groups.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id is not None and not df[numeric_field_id].isnull().all():
    plt.figure(figsize=(8, 5))
    sns.histplot(df[numeric_field_id].dropna(), bins=30, kde=True)
    plt.title(f'Distribution of {numeric_field_id}')
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.tight_layout()
    plt.show()

    # If grouping field was found, make a barplot
    if 'group_field_id' in locals() and group_field_id and group_field_id in filtered_df.columns:
        plt.figure(figsize=(8,5))
        plot_data = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        sns.barplot(x=group_field_id, y=numeric_field_id, data=plot_data)
        plt.title(f'Mean {numeric_field_id} by {group_field_id}')
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()

## 6. Conclusion
We explored the FAIR² dataset using the `mlcroissant` library, referencing all entities by their `@id`. We loaded metadata, discovered the structure of all available record sets and fields, and extracted and processed the main data for analysis and visualization. For more advanced analyses, refer to the record set and field `@id`s for precise and reproducible operations on your data.

---